In [3]:
import warnings
warnings.filterwarnings("ignore")

In [15]:
import ewatercycle.forcing
import ewatercycle.observation.grdc
import ewatercycle.analysis
import xarray as xr
from pathlib import Path
from cartopy.io import shapereader
from forcing import HBVForcing
from ewatercycle_wrapper_HBV import HBV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rich import print
import shutil


In [ ]:
# ============================================================
# COMPLETE HBV eWaterCycle CALIBRATION SCRIPT
# For Manning catchment using:
#   - CHIRPS precipitation
#   - ERA5 evaporation
#   - GRDC discharge
# Period: 2014-01-01 to 2025-01-31


# ============================================================
# 1. FILE PATHS
# ============================================================
precip_file = "./Data/manning_chirps_precip_daily.csv"
evap_file   = "./Data/manning_ERA5_evap_daily.csv"
q_file      = "./Data/5202080_Q_Day.Cmd.txt"

# output forcing netcdf
forcing_nc = "./Data/manning_hbv_forcing_2014_2025.nc"

# model period
experiment_start_time = "2014-01-01T00:00:00Z"
experiment_end_time   = "2025-01-31T00:00:00Z"

# calibration period
calibration_start_time = "2015-01-01"
calibration_end_time   = "2021-12-31"

# validation period
validation_start_time = "2022-01-01"
validation_end_time   = "2025-01-31"

# catchment area
# GRDC header says 6642.0 km² for KILLAWARRA
shape_area_km2 = 6642.0
shape_area = shape_area_km2 * 1e6   # m²


# ============================================================
# 2. BUILD CUSTOM FORCING NETCDF FROM YOUR CSV FILES
#    This creates one NetCDF with variables:
#       - pr
#       - evspsblpot
# ============================================================

# precipitation
P = pd.read_csv(precip_file)
P["time"] = pd.to_datetime(P["system:index"], format="%Y%m%d")
P = P.set_index("time")

# adjust if your precipitation column has a different name
P = P[["precipitation"]].rename(columns={"precipitation": "pr"})

# evaporation
E = pd.read_csv(evap_file)
E["time"] = pd.to_datetime(E["system:index"], format="%Y%m%d")
E = E.set_index("time")

# adjust if your evaporation column has a different name
E = E[["total_evaporation_sum"]].rename(columns={"total_evaporation_sum": "evspsblpot"})

# ERA5 evaporation is often stored negative
E["evspsblpot"] = E["evspsblpot"].abs()

# merge and subset period
forcing_df = P.join(E, how="inner")
forcing_df = forcing_df.loc["2014-01-01":"2025-01-31"].copy()

print("Forcing period:")
print(forcing_df.index.min(), "to", forcing_df.index.max())
print("Number of days:", len(forcing_df))
print(forcing_df.head())

# convert to xarray dataset
ds_forcing = xr.Dataset(
    data_vars={
        "pr": ("time", forcing_df["pr"].values),
        "evspsblpot": ("time", forcing_df["evspsblpot"].values),
    },
    coords={"time": forcing_df.index.values},
    attrs={
        "title": "HBV forcing for Manning catchment",
        "history": "Created from CHIRPS daily precipitation and ERA5 daily evaporation"
    }
)

ds_forcing["pr"].attrs["units"] = "mm/day"
ds_forcing["evspsblpot"].attrs["units"] = "mm/day"

ds_forcing.to_netcdf(forcing_nc)
print(f"\nSaved forcing NetCDF to: {forcing_nc}")


# ============================================================
# 3. LOAD GRDC OBSERVATIONS
# ============================================================

obs = pd.read_csv(q_file, sep=";", comment="#")
obs.columns = [c.strip() for c in obs.columns]

obs["YYYY-MM-DD"] = pd.to_datetime(obs["YYYY-MM-DD"])
obs["Value"] = pd.to_numeric(obs["Value"], errors="coerce")

# GRDC missing value code
obs.loc[obs["Value"] == -999.000, "Value"] = np.nan

obs = obs.set_index("YYYY-MM-DD")[["Value"]].rename(columns={"Value": "Q"})
obs = obs.loc["2014-01-01":"2025-01-31"].copy()

print("\nObservation period:")
print(obs.index.min(), "to", obs.index.max())
print("Number of days:", len(obs))
print(obs.head())


# ============================================================
# 4. CREATE HBVForcing OBJECT
# ============================================================

ERA5_forcing = HBVForcing(
    directory=Path("./Data"),
    start_time=experiment_start_time,
    end_time=experiment_end_time,
    pr="manning_hbv_forcing_2014_2025.nc",
    evspsblpot="manning_hbv_forcing_2014_2025.nc"
)


# ============================================================
# 5. PARAMETER AND INITIAL STORAGE SETTINGS
#    Parameter names from your HBV wrapper:
#       Imax, Ce, Sumax, Beta, Pmax, Tlag, Kf, Ks
# ============================================================

# initial storages
# Si, Su, Sf, Ss
s_0 = np.array([0.0, 100.0, 5.0, 20.0])
s_names = ["Interception storage", "Unsaturated Rootzone Storage", "Fastflow storage", "Groundwater storage"]

# parameter names
p_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks"]

# ensemble size
N = 100

# parameter ranges
#                Imax   Ce   Sumax  Beta  Pmax  Tlag   Kf      Ks
p_min_initial = np.array([0.0,  0.4,  40.0,  1.0,  0.001,  1.0,  0.01,   0.0001])
p_max_initial = np.array([8.0,  0.8, 800.0,  2.5,  0.3,   8.0,  0.10,   0.01])

parameters = np.zeros((8, N))

np.random.seed(6)
for param in range(8):
    parameters[param, :] = np.random.uniform(
        p_min_initial[param],
        p_max_initial[param],
        N
    )

print("\nRandom parameter ensemble created with shape:", parameters.shape)


# ============================================================
# 6. CREATE ENSEMBLE OF MODELS
#    This follows the same structure as your example notebook
# ============================================================

ensemble = []

for counter in range(N):
    ensemble.append(HBV(forcing=ERA5_forcing))

    config_file, _ = ensemble[counter].setup(
        parameters=",".join([str(p) for p in parameters[:, counter]]),
        initial_storage=",".join([str(s) for s in s_0]),
        cfg_dir="configFiles/hbv_ensembleMember_" + str(counter),
    )

    ensemble[counter].initialize(config_file)

print(f"\nInitialized {N} HBV ensemble members.")


# ============================================================
# 7. OBJECTIVE FUNCTIONS
#    We calculate:
#       - NSE
#       - log NSE
# ============================================================

def calibrationObjective(modelOutput, observation, start_calibration, end_calibration):
    """
    Returns:
        nse_value, log_nse_value
    """

    hydro_data = pd.concat(
        [modelOutput.reindex(observation.index, method="ffill"), observation],
        axis=1
    )

    hydro_data = hydro_data[hydro_data.index >= pd.to_datetime(pd.Timestamp(start_calibration).date())]
    hydro_data = hydro_data[hydro_data.index <= pd.to_datetime(pd.Timestamp(end_calibration).date())]

    hydro_data = hydro_data.dropna()

    sim = hydro_data["model output"].values
    obs_vals = hydro_data["Q"].values

    # NSE
    denom = np.sum((obs_vals - np.mean(obs_vals)) ** 2)
    if denom == 0:
        nse_value = np.nan
    else:
        nse_value = 1.0 - np.sum((obs_vals - sim) ** 2) / denom

    # log NSE
    eps = 1e-6
    sim_log = np.log(sim + eps)
    obs_log = np.log(obs_vals + eps)

    denom_log = np.sum((obs_log - np.mean(obs_log)) ** 2)
    if denom_log == 0:
        log_nse_value = np.nan
    else:
        log_nse_value = 1.0 - np.sum((obs_log - sim_log) ** 2) / denom_log

    return nse_value, log_nse_value


# ============================================================
# 8. RUN ALL ENSEMBLE MEMBERS
#    Model Q is in mm/day
#    Convert to m³/s using:
#      Q(m3/s) = Q(mm/day) * area(m²) / (1000 * 86400)
# ============================================================

objectives = []
all_model_outputs = []

for ensembleMember in ensemble:
    Q_m = []
    time = []

    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        

        discharge_this_timestep = ensembleMember.get_value("Q") * shape_area / (1000.0 * 86400.0)

        Q_m.append(discharge_this_timestep[0])
        time.append(pd.Timestamp(ensembleMember.time_as_datetime.date()))

    discharge_dataframe = pd.DataFrame(
        {"model output": Q_m},
        index=pd.to_datetime(time)
    )

    objective_this_model = calibrationObjective(
        discharge_dataframe,
        obs,
        calibration_start_time,
        calibration_end_time
    )

    objectives.append(objective_this_model)
    all_model_outputs.append(discharge_dataframe)

    if USE_PROGRESS:
        f.value += 1

print("\nFinished all ensemble runs.")


# ============================================================
# 9. FINALIZE MODELS
# ============================================================

for ensembleMember in ensemble:
    ensembleMember.finalize()

print("All models finalized.")


# ============================================================
# 10. ANALYSE RESULTS
# ============================================================

nse_values = [i[0] for i in objectives]
log_nse_values = [i[1] for i in objectives]

best_nse_index = int(np.nanargmax(nse_values))
best_log_nse_index = int(np.nanargmax(log_nse_values))

best_nse_params = parameters[:, best_nse_index]
best_log_nse_params = parameters[:, best_log_nse_index]

print("\nBest NSE index:", best_nse_index)
print("Best NSE value:", nse_values[best_nse_index])
print("Best NSE parameters:")
for name, val in zip(p_names, best_nse_params):
    print(f"  {name:>5s} = {val:.6f}")

print("\nBest log NSE index:", best_log_nse_index)
print("Best log NSE value:", log_nse_values[best_log_nse_index])
print("Best log NSE parameters:")
for name, val in zip(p_names, best_log_nse_params):
    print(f"  {name:>5s} = {val:.6f}")


# ============================================================
# 11. PLOT PARAMETER VS NSE / log NSE
# ============================================================

xFigNr = 2
yFigNr = 4

fig_nse, axs_nse = plt.subplots(xFigNr, yFigNr, figsize=(15, 15))
fig_lognse, axs_lognse = plt.subplots(xFigNr, yFigNr, figsize=(15, 15))

for xFig in range(xFigNr):
    for yFig in range(yFigNr):
        paramCounter = xFig * yFigNr + yFig

        axs_nse[xFig, yFig].plot(parameters[paramCounter, :], nse_values, ".", label="NSE")
        axs_nse[xFig, yFig].set_title(p_names[paramCounter])
        axs_nse[xFig, yFig].legend()

        axs_lognse[xFig, yFig].plot(parameters[paramCounter, :], log_nse_values, ".", label="log NSE")
        axs_lognse[xFig, yFig].set_title(p_names[paramCounter])
        axs_lognse[xFig, yFig].legend()

plt.tight_layout()
plt.show()


# ============================================================
# 12. PLOT BEST CALIBRATION MEMBER AGAINST OBSERVATIONS
# ============================================================

best_model_output = all_model_outputs[best_nse_index].copy()
best_model_output = best_model_output.reindex(obs.index, method="ffill")

plot_df = pd.concat([obs, best_model_output], axis=1)
plot_df = plot_df.loc["2014-01-01":"2025-01-31"]

plt.figure(figsize=(16, 5))
plt.plot(plot_df.index, plot_df["Q"], label="Observed Q")
plt.plot(plot_df.index, plot_df["model output"], label="Best NSE model output")
plt.title("Observed vs Best NSE simulated discharge")
plt.ylabel("Discharge [m³/s]")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 13. CALCULATE NSE FOR VALIDATION PERIOD AS WELL
# ============================================================

validation_scores = calibrationObjective(
    best_model_output,
    obs,
    validation_start_time,
    validation_end_time
)

print("\nValidation performance of best-NSE member:")
print("Validation NSE    :", validation_scores[0])
print("Validation log NSE:", validation_scores[1])


# ============================================================
# 14. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame({
    "member": np.arange(N),
    "NSE": nse_values,
    "logNSE": log_nse_values,
    "Imax": parameters[0, :],
    "Ce": parameters[1, :],
    "Sumax": parameters[2, :],
    "Beta": parameters[3, :],
    "Pmax": parameters[4, :],
    "Tlag": parameters[5, :],
    "Kf": parameters[6, :],
    "Ks": parameters[7, :]
})

results_df.to_csv("./Data/hbv_ensemble_results_2014_2025.csv", index=False)
plot_df.to_csv("./Data/hbv_best_member_timeseries_2014_2025.csv")

print("\nSaved:")
print("./Data/hbv_ensemble_results_2014_2025.csv")
print("./Data/hbv_best_member_timeseries_2014_2025.csv")

Forcing period:

2014-01-01 00:00:00 to 2025-01-31 00:00:00

Number of days: 4049

pr  evspsblpot
time                            
2014-01-01  0.000000    0.003428
2014-01-02  0.000000    0.002923
2014-01-03  0.000000    0.003176
2014-01-04  0.542302    0.003262
2014-01-05  0.000000    0.003388

Saved forcing NetCDF to: ./Data/manning_hbv_forcing_2014_2025.nc

...

2014-01-01 00:00:00 to 2025-01-31 00:00:00

Number of days: 4049

Q
YYYY-MM-DD       
2014-01-01  3.837
2014-01-02  3.232
2014-01-03  2.674
2014-01-04  2.170
2014-01-05  1.788

Random parameter ensemble created with shape:
(8, 100)

Initialized 100 HBV ensemble members.